# 3D Graphic Designing Using Matrix Chain Multiplication
## Kaggle Dataset Notebook

This notebook loads `kaggle_dataset.csv`, explores 3D object parameters, builds 4×4 transformation matrices for translation, scaling, and rotation, and exports the calculated results.

**Columns expected:** `id`, `object_name`, `x`, `y`, `z`, `scale_x`, `scale_y`, `scale_z`, `rotation_x`, `rotation_y`, `rotation_z`.

In Kaggle, add `kaggle_dataset.csv` to the notebook's **Input** section. Locally, keep the CSV beside this notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [
    "/kaggle/input/kaggle-dataset/kaggle_dataset.csv",
    "/kaggle/input/kaggle_dataset.csv",
    "kaggle_dataset.csv",
    "/mnt/data/kaggle_dataset.csv",
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("Add kaggle_dataset.csv to Kaggle Notebook Input or place it beside this notebook.")

df = pd.read_csv(csv_path)
print(f"Loaded {csv_path} | shape={df.shape}")
display(df.head())

## 1. Validate and explore the dataset

In [ ]:
required = ["id", "object_name", "x", "y", "z", "scale_x", "scale_y", "scale_z", "rotation_x", "rotation_y", "rotation_z"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))
print("Summary statistics:")
display(df.describe(include="all").T)

## 2. Visualize 3D object positions

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(df["x"], df["y"], df["z"], s=90)
for _, row in df.iterrows():
    ax.text(row["x"], row["y"], row["z"], str(row["object_name"]))
ax.set_xlabel("X position")
ax.set_ylabel("Y position")
ax.set_zlabel("Z position")
ax.set_title("3D Object Positions from Dataset")
plt.tight_layout()
plt.show()

## 3. Build transformation matrices

For column-vector coordinates, the chosen model transformation is:

`Model = Translation × RotationZ × RotationY × RotationX × Scaling`

The multiplication order matters because matrix multiplication is not commutative. Homogeneous coordinates let translation, rotation, and scaling use 4×4 matrices.

In [ ]:
def translation_matrix(x, y, z):
    return np.array([[1, 0, 0, x], [0, 1, 0, y], [0, 0, 1, z], [0, 0, 0, 1]], dtype=float)

def scaling_matrix(sx, sy, sz):
    return np.diag([sx, sy, sz, 1.0])

def rotation_x_matrix(degrees):
    a = np.deg2rad(degrees)
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0, 0], [0, c, -s, 0], [0, s, c, 0], [0, 0, 0, 1]], dtype=float)

def rotation_y_matrix(degrees):
    a = np.deg2rad(degrees)
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s, 0], [0, 1, 0, 0], [-s, 0, c, 0], [0, 0, 0, 1]], dtype=float)

def rotation_z_matrix(degrees):
    a = np.deg2rad(degrees)
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0, 0], [s, c, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)

def matrix_chain_order(dimensions):
    """Return minimum multiplication cost and split table using dynamic programming."""
    n = len(dimensions) - 1
    cost = [[0] * n for _ in range(n)]
    split = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            cost[i][j] = float("inf")
            for k in range(i, j):
                q = cost[i][k] + cost[k + 1][j] + dimensions[i] * dimensions[k + 1] * dimensions[j + 1]
                if q < cost[i][j]:
                    cost[i][j], split[i][j] = q, k
    return cost, split

def parenthesize(split, i, j):
    if i == j:
        return f"M{i + 1}"
    k = split[i][j]
    return f"({parenthesize(split, i, k)} × {parenthesize(split, k + 1, j)})"

def build_model_matrix(row):
    T = translation_matrix(row.x, row.y, row.z)
    Rz = rotation_z_matrix(row.rotation_z)
    Ry = rotation_y_matrix(row.rotation_y)
    Rx = rotation_x_matrix(row.rotation_x)
    S = scaling_matrix(row.scale_x, row.scale_y, row.scale_z)
    matrices = [T, Rz, Ry, Rx, S]
    dimensions = [4, 4, 4, 4, 4, 4]
    costs, splits = matrix_chain_order(dimensions)
    # DP chooses the cheapest parenthesization; multiplication order remains T @ Rz @ Ry @ Rx @ S.
    expression = parenthesize(splits, 0, len(matrices) - 1)
    model = T @ Rz @ Ry @ Rx @ S
    return model, int(costs[0][-1]), expression

results = []
model_matrices = {}
for row in df.itertuples(index=False):
    model, min_cost, expression = build_model_matrix(row)
    model_matrices[row.object_name] = model
    results.append({
        "object_name": row.object_name,
        "min_scalar_multiplications_for_five_4x4_matrices": min_cost,
        "optimal_parenthesization": expression,
        "translated_x": model[0, 3],
        "translated_y": model[1, 3],
        "translated_z": model[2, 3],
    })

transformed_df = pd.DataFrame(results)
display(transformed_df.round(4))

## 4. Inspect an object's 4×4 model matrix

In [ ]:
# Change this to any object_name present in your CSV.
selected_object = df.loc[0, "object_name"]
print("Selected object:", selected_object)
display(pd.DataFrame(model_matrices[selected_object],
                     index=["row_1", "row_2", "row_3", "row_4"],
                     columns=["col_1", "col_2", "col_3", "col_4"]))

## 5. Export results

In [ ]:
output_path = "transformed_3d_objects.csv"
transformed_df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")
display(transformed_df)

## Conclusion
The notebook uses the supplied CSV as its input rather than fixed outputs. Editing an object's position, scale, or rotation in the dataset changes the calculated model matrix. The dynamic-programming section calculates a minimum-cost parenthesization for multiplying the five 4×4 matrices; it does not change the required transformation order.